In [16]:
# ==========================================================
# Aircraft Metadata Feature Engineering
# Dissertation Project
# ==========================================================

import pandas as pd
import numpy as np
from datetime import datetime

# Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [17]:
# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("../../data/processed/aircraft/aircraft_metadata_final.csv")

print("Dataset Loaded Successfully")
print("-" * 50)
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

df.head()

Dataset Loaded Successfully
--------------------------------------------------
Rows : 314417
Columns : 29


,N-NUMBER,SERIAL NUMBER,MFR MDL CODE,ENG MFR MDL,YEAR MFR,TYPE REGISTRANT,LAST ACTION DATE,CERT ISSUE DATE,CERTIFICATION,TYPE AIRCRAFT,TYPE ENGINE,STATUS CODE,MODE S CODE,AIR WORTH DATE,EXPIRATION DATE,UNIQUE ID,MODE S CODE HEX,MANUFACTURER,AIRCRAFT_MODEL,AIRCRAFT_TYPE_CODE,ENGINE_TYPE_CODE,AIRCRAFT_CATEGORY,BUILD_CERTIFICATE,NUMBER_OF_ENGINES,SEAT_CAPACITY,WEIGHT_CLASS,CRUISE_SPEED,TYPE_CERTIFICATE,CERTIFICATE_HOLDER
0,100,5334,7100510,17003.0,1940.0,1.0,2023-01-22,2005-05-06,1,4,1,V,50002263,1954-04-30,2027-04-30,600060,A004B3,PIPER,J3C-65,4,1,1,0,1,2,CLASS 1,67,,...
1,10000,10000,2130004,NaN,NaN,7.0,2024-08-23,2024-08-23,NaN,4,1,V,50003445,NaN,2031-08-31,1443200,A00725,CIRRUS DESIGN CORP,SR22T,4,1,1,0,1,5,CLASS 1,0,,...
2,10001,A28,9601202,67007.0,1928.0,1.0,2023-07-18,2019-02-27,1,4,1,V,50003446,NaN,2029-02-28,432072,A00726,WACO,ASO,4,1,1,0,1,3,CLASS 1,79,,...
3,10004,T18208245,2072738,NaN,NaN,7.0,2023-07-22,2013-03-12,NaN,4,2,V,50003451,NaN,2029-03-31,102879,A00729,CESSNA,T182T,4,2,1,0,1,4,CLASS 1,0,3A13,TEXTRON AVIATION INC ...
4,10006,BG-72,1152020,17026.0,1955.0,1.0,2023-04-21,1998-08-26,1U,4,1,V,50003453,1971-09-09,2028-02-29,480110,A0072B,BEECH,D-45 (T-34B),4,1,1,0,1,4,CLASS 1,0,,...


In [18]:
# ==========================================================
# Dataset Information
# ==========================================================

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 314417 entries, 0 to 314416
Data columns (total 29 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   N-NUMBER            314417 non-null  str    
 1   SERIAL NUMBER       314417 non-null  str    
 2   MFR MDL CODE        314417 non-null  str    
 3   ENG MFR MDL         276838 non-null  float64
 4   YEAR MFR            246006 non-null  float64
 5   TYPE REGISTRANT     313225 non-null  float64
 6   LAST ACTION DATE    314417 non-null  str    
 7   CERT ISSUE DATE     306526 non-null  str    
 8   CERTIFICATION       280514 non-null  str    
 9   TYPE AIRCRAFT       314417 non-null  str    
 10  TYPE ENGINE         314417 non-null  int64  
 11  STATUS CODE         314417 non-null  str    
 12  MODE S CODE         314417 non-null  int64  
 13  AIR WORTH DATE      272438 non-null  str    
 14  EXPIRATION DATE     306494 non-null  str    
 15  UNIQUE ID           314417 non-null  int64  


In [19]:
# ==========================================================
# Summary Statistics
# ==========================================================

df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
N-NUMBER,314417,314417,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SERIAL NUMBER,314417,246556,001,1764,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MFR MDL CODE,314417,44983,7102802,4264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENG MFR MDL,276838.0,NaN,NaN,NaN,34460.436118,18986.097364,0.0,17026.0,41508.0,41678.0,99999.0
YEAR MFR,246006.0,NaN,NaN,NaN,1984.467363,24.154516,1909.0,1966.0,1979.0,2006.0,2026.0
TYPE REGISTRANT,313225.0,NaN,NaN,NaN,3.417112,2.495795,1.0,1.0,3.0,7.0,9.0
LAST ACTION DATE,314417,1970,2023-01-22,12214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CERT ISSUE DATE,306526,13483,2026-06-12,273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CERTIFICATION,280514,313,1N,97923,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TYPE AIRCRAFT,314417,11,4,220963,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# ==========================================================
# Missing Values
# ==========================================================

missing = (
    df.isnull()
      .sum()
      .reset_index()
)

missing.columns = ["Column", "Missing Values"]

missing["Percentage"] = (
    missing["Missing Values"] / len(df) * 100
).round(2)

missing.sort_values(
    "Percentage",
    ascending=False,
    inplace=True
)

missing

,Column,Missing Values,Percentage
4,YEAR MFR,68411,21.76
13,AIR WORTH DATE,41979,13.35
3,ENG MFR MDL,37579,11.95
8,CERTIFICATION,33903,10.78
14,EXPIRATION DATE,7923,2.52
7,CERT ISSUE DATE,7891,2.51
5,TYPE REGISTRANT,1192,0.38
1,SERIAL NUMBER,0,0.00
2,MFR MDL CODE,0,0.00
9,TYPE AIRCRAFT,0,0.00


In [21]:
# ==========================================================
# Inspect Source Columns
# ==========================================================

columns = [

    "YEAR MFR",
    "NUMBER_OF_ENGINES",
    "SEAT_CAPACITY",
    "WEIGHT_CLASS",
    "CRUISE_SPEED",
    "MANUFACTURER",
    "AIRCRAFT_TYPE_CODE"

]

for column in columns:

    print("\n" + "="*70)
    print(column)
    print("="*70)

    print(df[column].value_counts(dropna=False).head(20))


YEAR MFR
YEAR MFR
NaN       68411
1946.0     8595
1978.0     6838
1979.0     6441
1977.0     6419
1976.0     6331
1966.0     5723
1975.0     5296
1968.0     4998
1974.0     4869
2007.0     4746
1973.0     4740
1967.0     4729
1965.0     4362
2006.0     4310
1969.0     4266
2008.0     4183
2005.0     3921
2023.0     3867
1981.0     3846
Name: count, dtype: int64

NUMBER_OF_ENGINES
NUMBER_OF_ENGINES
1     238092
2      50219
0       9134
4       7953
8       3606
5       2843
6       1390
3       1094
9         22
12        18
10        15
16        12
7          5
20         5
13         3
80         2
32         1
33         1
18         1
14         1
Name: count, dtype: int64

SEAT_CAPACITY
SEAT_CAPACITY
4      101190
2       87129
6       27042
1       17545
0       17202
5       10391
8        8576
7        7891
3        4935
11       4107
12       3205
10       2970
9        2599
22       1921
19       1587
15       1259
13        936
55        834
14        815
175       737
Nam

In [22]:
# ==========================================================
# Feature Engineering 1: Aircraft Age
# ==========================================================

# Convert manufacturing year to numeric
df["YEAR MFR"] = pd.to_numeric(df["YEAR MFR"], errors="coerce")

# Current year
CURRENT_YEAR = datetime.now().year

# Calculate aircraft age
df["Aircraft_Age"] = CURRENT_YEAR - df["YEAR MFR"]

# Remove impossible ages
df.loc[
    (df["Aircraft_Age"] < 0) |
    (df["Aircraft_Age"] > 120),
    "Aircraft_Age"
] = np.nan

print("Aircraft Age Feature Created Successfully")

Aircraft Age Feature Created Successfully


In [23]:
# ==========================================================
# Aircraft Age Summary
# ==========================================================

print(df["Aircraft_Age"].describe())

print("\nMissing Values:", df["Aircraft_Age"].isna().sum())

count    246006.000000
mean         41.532637
std          24.154516
min           0.000000
25%          20.000000
50%          47.000000
75%          60.000000
max         117.000000
Name: Aircraft_Age, dtype: float64

Missing Values: 68411


In [24]:
# Distribution of Aircraft Age

df["Aircraft_Age"].value_counts().sort_index().head(20)

Aircraft_Age
0.0     1155
1.0     3707
2.0     3805
3.0     3867
4.0     3326
5.0     2919
6.0     2687
7.0     3100
8.0     2863
9.0     2805
10.0    2762
11.0    2895
12.0    2837
13.0    2641
14.0    2600
15.0    2342
16.0    2305
17.0    2581
18.0    4183
19.0    4746
Name: count, dtype: int64

In [25]:
# ==========================================================
# Feature Engineering 2: Aircraft Age Category
# ==========================================================

def aircraft_age_category(age):

    if pd.isna(age):
        return "Unknown"

    elif age <= 5:
        return "Very New"

    elif age <= 15:
        return "Modern"

    elif age <= 30:
        return "Mid-Life"

    elif age <= 50:
        return "Aging"

    else:
        return "Legacy"


df["Aircraft_Age_Category"] = df["Aircraft_Age"].apply(aircraft_age_category)

print("Aircraft Age Category Created Successfully")

Aircraft Age Category Created Successfully


In [26]:
# Aircraft Age Category Distribution

df["Aircraft_Age_Category"].value_counts()

Aircraft_Age_Category
Legacy      99617
Unknown     68411
Aging       51912
Mid-Life    48166
Modern      27532
Very New    18779
Name: count, dtype: int64

In [27]:
# ==========================================================
# Feature Engineering 3: Engine Configuration
# ==========================================================

def engine_configuration(engines):

    if pd.isna(engines):
        return "Unknown"

    elif engines == 0:
        return "Unknown"

    elif engines == 1:
        return "Single Engine"

    elif engines == 2:
        return "Twin Engine"

    elif engines == 3:
        return "Three Engine"

    else:
        return "Multi Engine"


df["Engine_Configuration"] = df["NUMBER_OF_ENGINES"].apply(engine_configuration)

print("Engine Configuration Feature Created Successfully")

Engine Configuration Feature Created Successfully


In [28]:
# ==========================================================
# Engine Configuration Distribution
# ==========================================================

df["Engine_Configuration"].value_counts()

Engine_Configuration
Single Engine    238092
Twin Engine       50219
Multi Engine      15878
Unknown            9134
Three Engine       1094
Name: count, dtype: int64

In [29]:
# ==========================================================
# Feature Engineering 4: Seat Capacity Category
# ==========================================================

def seat_capacity_category(seats):

    if pd.isna(seats):
        return "Unknown"

    elif seats == 0:
        return "Unknown"

    elif seats <= 6:
        return "Very Small"

    elif seats <= 19:
        return "Small"

    elif seats <= 99:
        return "Regional"

    elif seats <= 199:
        return "Medium Commercial"

    else:
        return "Large Commercial"


df["Seat_Capacity_Category"] = df["SEAT_CAPACITY"].apply(seat_capacity_category)

print("Seat Capacity Category Created Successfully")

Seat Capacity Category Created Successfully


In [30]:
# ==========================================================
# Seat Capacity Category Distribution
# ==========================================================

df["Seat_Capacity_Category"].value_counts()

Seat_Capacity_Category
Very Small           248232
Small                 34735
Unknown               17202
Regional               7266
Medium Commercial      4356
Large Commercial       2626
Name: count, dtype: int64

In [31]:
# ==========================================================
# Feature Engineering 5: Weight Category
# ==========================================================

weight_mapping = {
    "CLASS 1": "Light",
    "CLASS 2": "Medium",
    "CLASS 3": "Heavy",
    "CLASS 4": "Very Heavy"
}

df["Weight_Category"] = (
    df["WEIGHT_CLASS"]
    .str.strip()
    .map(weight_mapping)
    .fillna("Unknown")
)

print("Weight Category Feature Created Successfully")

Weight Category Feature Created Successfully


In [32]:
# ==========================================================
# Weight Category Distribution
# ==========================================================

df["Weight_Category"].value_counts()

Weight_Category
Light         281667
Heavy          19188
Medium         10406
Very Heavy      3156
Name: count, dtype: int64

In [33]:
# ==========================================================
# Feature Engineering 6: Cruise Speed Category
# ==========================================================

def cruise_speed_category(speed):

    if pd.isna(speed):
        return "Unknown"

    elif speed == 0:
        return "Unknown"

    elif speed <= 120:
        return "Low Speed"

    elif speed <= 250:
        return "Medium Speed"

    elif speed <= 450:
        return "High Speed"

    else:
        return "Very High Speed"


df["Cruise_Speed_Category"] = df["CRUISE_SPEED"].apply(cruise_speed_category)

print("Cruise Speed Category Created Successfully")

Cruise Speed Category Created Successfully


In [34]:
# ==========================================================
# Cruise Speed Category Distribution
# ==========================================================

df["Cruise_Speed_Category"].value_counts()

Cruise_Speed_Category
Unknown            160354
Low Speed          111146
Medium Speed        41872
High Speed           1017
Very High Speed        28
Name: count, dtype: int64

In [35]:
# ==========================================================
# Feature Engineering 7: Manufacturer Group
# ==========================================================

def manufacturer_group(manufacturer):

    if pd.isna(manufacturer):
        return "Unknown"

    manufacturer = str(manufacturer).upper()

    # Commercial Aircraft
    if "BOEING" in manufacturer or "AIRBUS" in manufacturer:
        return "Commercial"

    # Regional Aircraft
    elif "EMBRAER" in manufacturer or "BOMBARDIER" in manufacturer:
        return "Regional"

    # Helicopters
    elif ("ROBINSON" in manufacturer or
          "BELL" in manufacturer or
          "SIKORSKY" in manufacturer or
          "MD HELICOPTERS" in manufacturer):
        return "Helicopter"

    # General Aviation
    elif ("CESSNA" in manufacturer or
          "PIPER" in manufacturer or
          "BEECH" in manufacturer or
          "BEECHCRAFT" in manufacturer or
          "CIRRUS" in manufacturer or
          "DIAMOND" in manufacturer or
          "MOONEY" in manufacturer):
        return "General Aviation"

    else:
        return "Other"


df["Manufacturer_Group"] = df["MANUFACTURER"].apply(manufacturer_group)

print("Manufacturer Group Feature Created Successfully")

Manufacturer Group Feature Created Successfully


In [36]:
# ==========================================================
# Manufacturer Group Distribution
# ==========================================================

df["Manufacturer_Group"].value_counts()

Manufacturer_Group
General Aviation    155806
Other               131506
Helicopter           11551
Commercial           10044
Regional              5510
Name: count, dtype: int64

In [37]:
# ==========================================================
# Feature Engineering 8: Aircraft Size Category
# ==========================================================

def aircraft_size_category(seats):

    if pd.isna(seats):
        return "Unknown"

    elif seats == 0:
        return "Unknown"

    elif seats <= 6:
        return "Small Aircraft"

    elif seats <= 19:
        return "Light Aircraft"

    elif seats <= 99:
        return "Regional Aircraft"

    elif seats <= 199:
        return "Narrow-Body"

    else:
        return "Wide-Body"


df["Aircraft_Size_Category"] = df["SEAT_CAPACITY"].apply(aircraft_size_category)

print("Aircraft Size Category Created Successfully")

Aircraft Size Category Created Successfully


In [38]:
# ==========================================================
# Aircraft Size Category Distribution
# ==========================================================

df["Aircraft_Size_Category"].value_counts()

Aircraft_Size_Category
Small Aircraft       248232
Light Aircraft        34735
Unknown               17202
Regional Aircraft      7266
Narrow-Body            4356
Wide-Body              2626
Name: count, dtype: int64

In [39]:
# ==========================================================
# Engineered Features
# ==========================================================

engineered_features = [

    "Aircraft_Age",
    "Aircraft_Age_Category",
    "Engine_Configuration",
    "Seat_Capacity_Category",
    "Weight_Category",
    "Cruise_Speed_Category",
    "Manufacturer_Group",
    "Aircraft_Size_Category"

]

df[engineered_features].head()

,Aircraft_Age,Aircraft_Age_Category,Engine_Configuration,Seat_Capacity_Category,Weight_Category,Cruise_Speed_Category,Manufacturer_Group,Aircraft_Size_Category
0,86.0,Legacy,Single Engine,Very Small,Light,Low Speed,General Aviation,Small Aircraft
1,NaN,Unknown,Single Engine,Very Small,Light,Unknown,General Aviation,Small Aircraft
2,98.0,Legacy,Single Engine,Very Small,Light,Low Speed,Other,Small Aircraft
3,NaN,Unknown,Single Engine,Very Small,Light,Unknown,General Aviation,Small Aircraft
4,71.0,Legacy,Single Engine,Very Small,Light,Unknown,General Aviation,Small Aircraft


In [40]:
# ==========================================================
# Save Feature Engineered Dataset
# ==========================================================

output_path = "../../data/processed/aircraft/aircraft_metadata_feature_engineered.csv"

df.to_csv(output_path, index=False)

print("Feature-engineered dataset saved successfully.")
print(f"Saved to: {output_path}")

Feature-engineered dataset saved successfully.
Saved to: ../../data/processed/aircraft/aircraft_metadata_feature_engineered.csv
